# 06. Baselines y modelado inicial

**Fases del guía metodológica cubiertas: 11 (Baselines), 12 (Modelado inicial)**

> Regla central aplicada: *definir -> auditar -> dividir -> aprender solo con train ->
> seleccionar con validación/CV -> comprobar una vez con test -> empaquetar -> monitorizar*.
> El test nunca influye en preprocessing, selección de variables, hiperparámetros o elección de modelo.



## 11. Baselines

Antes de modelos complejos, medimos referencias simples **con el mismo protocolo de CV**
(5 folds estratificados sobre train, pipeline dentro de cada fold):

1. **Dummy estratificado** (clase aleatoria con la prevalencia real) -> referencia inferior.
2. **Regla de negocio** (heurística `goout>=4 AND Dalc>=3`) -> referencia de coste cero.

### 11.0.1 Preparación de los datos

Cargamos los conjuntos y aplicamos el feature engineering. A partir de aquí, todos los
experimentos usan exactamente las mismas particiones (guardadas en la fase 7), de modo
que las comparaciones entre modelos sean justas: cualquier diferencia de métrica se debe
al modelo, no a la muestra.


In [1]:

import sys, pathlib
ROOT = pathlib.Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.load_data import load_processed
from src.features.build_features import add_domain_features
d = load_processed()
Xtr = add_domain_features(d["X_train"]); ytr = d["y_train"]
Xva = add_domain_features(d["X_val"]);   yva = d["y_val"]
Xte = add_domain_features(d["X_test"]);  yte = d["y_test"]
print("OK", Xtr.shape, Xva.shape, Xte.shape)


OK (396, 37) (133, 37) (133, 37)



### 11.0.2 Dummy estratificado y regla de negocio

Entrenamos el `DummyClassifier` (estrategia `stratified`) sobre train y evaluamos su
ROC-AUC en validation: debe rondar 0.5, el rendimiento de "adivinar al azar con la
prevalencia real". Después aplicamos la **regla de negocio** (`goout >= 4` y `Dalc >= 3`)
directamente sobre validation: sin entrenar nada, con una sola condición legible,
obtenemos una primera referencia realista. Cualquier modelo de ML debe superar ambas
referencias para justificar su complejidad.


In [2]:

# Baseline 1: Dummy estratificado + Baseline 2: regla de negocio
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score

dummy = DummyClassifier(strategy="stratified", random_state=42)
dummy.fit(Xtr, ytr)
print("Dummy ROC-AUC (val):", round(roc_auc_score(yva, dummy.predict_proba(Xva)[:, 1]), 4))

# Regla de negocio: salir mucho + consumo entre semana alto
regla = ((Xva["goout"] >= 4) & (Xva["Dalc"] >= 3)).astype(int)
print("Regla de negocio ROC-AUC (val):", round(roc_auc_score(yva, regla), 4))
print("Regla de negocio precisión:", round((regla == yva).mean(), 4))


Dummy ROC-AUC (val): 0.5133
Regla de negocio ROC-AUC (val): 0.5769
Regla de negocio precisión: 0.6692



## 12. Secuencia de complejidad

`Baseline -> LogReg -> KNN -> Árbol -> RandomForest -> XGBoost -> LightGBM -> CatBoost`

### 12.0.1 Validación cruzada de los 7 candidatos

Comparamos las 7 familias con **StratifiedKFold(5)** sobre train. Cada fold entrena un
pipeline **completo** (preprocessor + modelo) ajustado solo con el sub-train del fold,
de modo que la codificación one-hot/ordinal nunca ve datos de validación (fase 13.3).
Imprimimos la media y desviación del ROC-AUC por modelo, junto con el tiempo de
entrenamiento. Esto nos da el primer ranking de candidatos.


In [3]:

# CV de los 7 candidatos (pipeline dentro de cada fold)
from src.models.train_model import get_model_factories, run_cv
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []
for name, factory in get_model_factories().items():
    if name == "BaselineDummy":
        continue
    r = run_cv(Xtr, ytr, name, factory, cv, metric="roc_auc")
    results.append(r)
    print(f"{name:20s} ROC-AUC CV = {r['mean']:.4f} +/- {r['std']:.4f}  ({r['time_s']}s)")

res_df = pd.DataFrame(results).sort_values("mean", ascending=False)
res_df


LogisticRegression   ROC-AUC CV = 0.8255 +/- 0.0224  (0.43s)


KNN                  ROC-AUC CV = 0.7403 +/- 0.0389  (4.37s)


DecisionTree         ROC-AUC CV = 0.7520 +/- 0.0189  (0.23s)


RandomForest         ROC-AUC CV = 0.8268 +/- 0.0440  (2.96s)


XGBoost              ROC-AUC CV = 0.8247 +/- 0.0532  (3.98s)


LightGBM             ROC-AUC CV = 0.8237 +/- 0.0426  (4.5s)


CatBoost             ROC-AUC CV = 0.8299 +/- 0.0526  (2.74s)


,model,metric,scores,mean,std,time_s
6,CatBoost,roc_auc,"[0.8873697916666666, 0.8487903225806452, 0.846...",0.829893,0.052650,2.74
3,RandomForest,roc_auc,"[0.8326822916666666, 0.8763440860215055, 0.826...",0.826752,0.043968,2.96
0,LogisticRegression,roc_auc,"[0.8444010416666666, 0.8225806451612904, 0.832...",0.825466,0.022441,0.43
4,XGBoost,roc_auc,"[0.8743489583333334, 0.8622311827956989, 0.833...",0.824735,0.053214,3.98
5,LightGBM,roc_auc,"[0.8658854166666666, 0.846774193548387, 0.8185...",0.823715,0.042631,4.50
2,DecisionTree,roc_auc,"[0.7659505208333334, 0.7570564516129032, 0.743...",0.751980,0.018854,0.23
1,KNN,roc_auc,"[0.7613932291666666, 0.7694892473118279, 0.739...",0.740316,0.038863,4.37



### 12.0.2 Registro de experimentos y figura comparativa

Guardamos el ranking en `reports/experiments.csv` (disciplina experimental de la fase
12.3: cada experimento queda registrado con su métrica, desviación y tiempo) y dibujamos
un **gráfico de barras con barras de error** (media +/- desviación) para visualizar qué
familias son competitivas y cuáles se quedan atrás. Esta figura se exporta a
`reports/figures/06_cv_modelos.png`.


In [4]:

# Guardar registro de experimentos (fase 12.3: disciplina experimental)
import json
records = []
for _, row in res_df.iterrows():
    records.append({"modelo": row["model"], "cv_mean": round(row["mean"], 4),
                    "cv_std": round(row["std"], 4), "tiempo_s": row["time_s"]})
(ROOT / "reports" / "experiments.csv").write_text(
    "modelo,cv_mean,cv_std,tiempo_s\n" + "\n".join(
        f"{r['modelo']},{r['cv_mean']},{r['cv_std']},{r['tiempo_s']}" for r in records),
    encoding="utf-8")

# Figura comparativa
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 4))
res_df.set_index("model")["mean"].plot.bar(ax=ax, yerr=res_df.set_index("model")["std"], capsize=4)
ax.set_ylabel("ROC-AUC (CV 5 folds)"); ax.set_ylim(0.4, 1.0)
plt.tight_layout(); plt.savefig(ROOT / "reports" / "figures" / "06_cv_modelos.png", dpi=120)
plt.show()


C:\Users\sgml1\AppData\Local\Temp\ipykernel_15328\907605996.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
